# Opportunity Layer Construction

## Overview

This notebook constructs the Opportunity Layer of the mobility modelling framework. It uses processed POI category counts derived from the Ordnance Survey Points of Interest (OS POI) dataset to represent the relative spatial opportunity of destination categories.

The notebook applies log transformation and renormalisation to produce category-level opportunity distributions for ambiguous trip purposes, which can subsequently be integrated into the framework.

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |District_manchester_poi_counts|
| Source | Ordnance Survey - Points of Interest Documentation |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |District_manchester_poi_counts|
| Output File | |
| Notebook Author | Peter |
| Last Updated |2026-07-24|

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## 2. Set Up File Pathes

In [2]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
project_root = Path.cwd().parents[1]

input_file_path = project_root / "data" / "POI" / "processed" / "manchester" / "District_manchester_poi_counts.xlsx"

## 3. Load Processed POI Data

In [4]:
poi_counts = pd.read_excel(input_file_path)

print(f"Dataset loaded successfully.")
print(f"Number of POI categories: {len(poi_counts)}")

Dataset loaded successfully.
Number of POI categories: 10


## 4. Data Inspection

In [5]:
display(poi_counts.head(10))

,POI Category,POI Count
0,Accommodation,139
1,Education,618
2,Employment,6031
3,Food and Beverage,2731
4,Healthcare,942
5,Personal Services,1613
6,Public Services,2899
7,Recreation,1256
8,Retail,2856
9,Transport,3401


## 5. Data Processing

### 5.1 Logarithmic Transformation of POI Counts

The Opportunity Layer assumes that areas containing more POIs provide greater opportunities for activities. However, raw POI counts are highly skewed across categories, with certain categories (e.g., Employment or Retail) containing substantially more POIs than others.

To reduce the dominance of extremely large counts while preserving their relative ordering, a logarithmic transformation is applied using the natural logarithm:

\[
\log(1+x)
\]

where \(x\) is the original POI count.

The transformed values are subsequently used to construct opportunity distributions.

In [12]:
poi_counts_processed = poi_counts.copy()


def log_transform_poi_counts(poi_counts_df):
    """
    Apply log(1 + x) transformation to POI counts.
    """

    df = poi_counts_df.copy()

    df["Log Count"] = np.log1p(df["POI Count"])

    return df


poi_counts_processed = log_transform_poi_counts(poi_counts_processed)

display(poi_counts_processed)

,POI Category,POI Count,Log Count
0,Accommodation,139,4.941642
1,Education,618,6.428105
2,Employment,6031,8.704834
3,Food and Beverage,2731,7.912789
4,Healthcare,942,6.849066
5,Personal Services,1613,7.386471
6,Public Services,2899,7.972466
7,Recreation,1256,7.136483
8,Retail,2856,7.957527
9,Transport,3401,8.132119


## 6. Opportunity Distribution Construction

### 6.1 Trip Purpose-to-POI Mapping

Trip purposes generated by the Behaviour Layer are first mapped to one or more candidate POI categories.

Most trip purposes correspond directly to a single POI category (e.g., Shopping → Retail). However, certain trip purposes, such as Personal Business, may reasonably occur across multiple POI categories.

The mapping below defines the candidate POI categories used when constructing opportunity distributions.

In [15]:
from ontology.TRIP_PURPOSE_TO_POI_MAPPING import TRIP_PURPOSE_TO_POI_MAPPING

TRIP_PURPOSE_TO_POI_MAPPING

,Unified Trip Purpose,Mapping Strategy,POI Categories
0,Commuting,Representative Mapping,[Employment]
1,Business,Representative Mapping,[Employment]
2,Visiting Friends Elsewhere,Representative Mapping,[Food and Beverage]
3,Entertainment or Public Activity,Representative Mapping,[Recreation]
4,Day Trip,Representative Mapping,[Recreation]
5,Just Walk,Representative Mapping,[Recreation]
6,Education,Direct Mapping,[Education]
7,Escort Education,Direct Mapping,[Education]
8,Shopping,Direct Mapping,[Retail]
9,Sport: Participate,Direct Mapping,[Recreation]
